# Ders 4A — RandomForest Sampling Gatekeeper Üzerinde Random Search Tuning

Step 6 sonucunda küçük de olsa ROC-AUC artışı veren sampling senaryoları yeni gatekeeper olarak alınır.

Bu notebookta:

- Sadece `RandomForest` kullanılır.
- Random search ile aday hiperparametre kombinasyonları üretilir.
- Her aday model sampling uygulanmış train set üzerinde yeniden eğitilir.
- Adayların performansı doğal test set üzerinde ayrı ayrı raporlanır.
- İç CV skorları ana sonuç tablosunda gösterilmez.
- Her pipeline için aday tablosu ve bar chart ayrı ayrı verilir.
- Final karar tablosunda sampling gatekeeper ile en iyi test ROC-AUC veren RF adayı karşılaştırılır.


## 1. Paket kontrolü

Bu scriptte gerçekten kullanılacak paketler kontrol edilir. Gereksiz paket import edilmez.

Bu dosyada yalnızca RandomForest, SimpleImputer ve RandomizedSearchCV kullanılır.

In [ ]:
import sys
# Mevcut Python ortamında pip komutunu çalıştırmak için kullanılır.
import subprocess
# Eksik paketleri notebook içinden kurmak için kullanılır.
import importlib.util
# Bir paketin kurulu olup olmadığını kontrol etmek için kullanılır.

def install_if_missing(import_name, pip_name=None):
    """Eksik paket varsa kurar; paket zaten varsa işlem yapmaz."""
    pip_name = pip_name or import_name
    # pip adı verilmezse import adı paket adı olarak kullanılır.
    if importlib.util.find_spec(import_name) is None:
        # Paket kurulu değilse pip kurulumu yapılır.
        print(f"[INSTALL] {pip_name}")
        # Hangi paketin kurulacağı ekrana yazdırılır.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Paket sessiz modda kurulur.
    else:
        print(f"[OK] {import_name}")
        # Paket zaten kuruluysa tekrar kurulmaz.

required_packages = [
    ("pandas", "pandas"),
    # CSV okuma ve tablo işlemleri için kullanılır.
    ("numpy", "numpy"),
    # Sayısal matris ve vektör işlemleri için kullanılır.
    ("matplotlib", "matplotlib"),
    # Grafik çizimleri için kullanılır.
    ("sklearn", "scikit-learn"),
    # RandomForest, RandomizedSearchCV ve metrikler için kullanılır.
    ("scipy", "scipy"),
    # Random search parametre dağılımları için kullanılır.
]
# Bu scriptte gerçekten kullanılan paketler listelenir.

for import_name, pip_name in required_packages:
    # Paketler tek tek kontrol edilir.
    install_if_missing(import_name, pip_name)
    # Eksik paket varsa kurulur.

print("Paket kontrolü tamamlandı.")
# Paket kontrolünün bittiği yazdırılır.

## 2. Importlar ve genel ayarlar

Bu hücrede yalnızca bu scriptte kullanılacak paketler ve sabitler çağırılır.

In [ ]:
from pathlib import Path
# Dosya ve klasör yollarını güvenli şekilde yönetmek için kullanılır.
import warnings
# Gereksiz uyarıları kontrol etmek için kullanılır.
warnings.filterwarnings("ignore")
# Notebook çıktısını sade tutmak için uyarılar gizlenir.

import numpy as np
# Feature matrisleri, sampling indexleri ve skor düzenleme işlemleri için kullanılır.
import pandas as pd
# Feature CSV dosyalarını okumak ve sonuç tablolarını kaydetmek için kullanılır.
import matplotlib.pyplot as plt
# Tuning sonuçlarını bar chart olarak çizmek için kullanılır.

from scipy.stats import randint
# RandomizedSearchCV için integer hiperparametre dağılımları sağlar.

from sklearn.model_selection import train_test_split, RandomizedSearchCV
# Aynı split mantığıyla train/test ayrımı ve random search tuning için kullanılır.
from sklearn.metrics import accuracy_score, balanced_accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
# Model performansını ölçmek için gerekli metrikler çağırılır.
from sklearn.ensemble import RandomForestClassifier
# Bu notebookta yalnızca RandomForest modeli kullanılacaktır.
from sklearn.impute import SimpleImputer
# Eksik değerleri train setten öğrenilen median değerlerle doldurmak için kullanılır.

DATASETS = {
    # İki veri setinin kısa isimleri ve GitHub raw feature dosyaları burada tanımlanır.
    "ERa_BLA_assay": {
        "short_name": "ERα BLA",
        "model_prefix": "model_ERa_BLA",
        "feature_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store/model_ERa_BLA_features.csv",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_1_A14_A17_ERa_BLA_agonist_antagonist.csv",
    },
    "ERa_LUC_VM7_assay": {
        "short_name": "ERα LUC VM7",
        "model_prefix": "model_ERa_LUC_VM7",
        "feature_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store/model_ERa_LUC_VM7_features.csv",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_2_A15_A18_ERa_LUC_VM7_agonist_antagonist.csv",
    },
}

RANDOM_STATE = 42
# Train/test ayrımı, RandomForest, sampling ve random search için sabit random seed belirlenir.
TEST_SIZE = 0.20
# Test set oranı %20 olarak belirlenir.
N_RANDOM_SEARCH_ITER = 10
# Random search 10 farklı hiperparametre kombinasyonu dener.
# Her kombinasyon önce 3-fold CV içinde değerlendirilir, sonra aynı adaylar doğal test set üzerinde ayrıca raporlanır.
# Veri seti başına 10 aday, iki veri seti için toplam 20 aday test performansı tablosuna girer.
CV_FOLDS = 3
# Random search içinde 3-fold CV kullanılır.

OUTPUT_ROOT = Path("molfest_outputs")
# Bu scriptin çıktılarının kaydedileceği ana klasör belirlenir.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Çıktı klasörü yoksa oluşturulur.

print("Ayarlar hazır.")
# Ayar hücresinin tamamlandığı yazdırılır.
print("Feature CSV klasörü:", "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store")
# Hazır feature dosyalarının okunacağı GitHub raw klasörü gösterilir.

## 3. Yardımcı fonksiyonlar

Bu fonksiyonlar notebook içinde görünür durumdadır ve sadece bu scriptte ihtiyaç duyulan işlemleri kapsar.

In [ ]:
def show_table(df, n=50, title=None):
    """Sonuç tablosunu Colab'da normal pandas tablo, terminalde metin olarak gösterir."""
    if title:
        # Tablo başlığı varsa önce başlık yazdırılır.
        print("\n" + title)
    table_df = df.head(n).copy()
    # Gösterilecek satır sayısı sınırlandırılır.
    table_df = table_df.reset_index(drop=True)
    # Tablo indexi temizlenir; böylece çıktı daha okunabilir olur.
    try:
        # Colab/Jupyter ortamında standart pandas tablo olarak gösterilir.
        display(table_df)
    except NameError:
        # Terminalde çalıştırılırsa tablo metin olarak yazdırılır.
        print(table_df.to_string(index=False))


def load_feature_table(dataset_key):
    """GitHub raw feature CSV dosyasını okur."""
    info = DATASETS[dataset_key]
    # Veri setine ait link ve kısa isim bilgileri alınır.
    df = pd.read_csv(info["feature_url"])
    # Hazır feature CSV dosyası GitHub raw linkinden okunur.
    print(f"\n{dataset_key} okundu: {df.shape[0]} satır, {df.shape[1]} kolon")
    # Okunan veri boyutu ekrana yazdırılır.
    return df
    # Okunan tablo döndürülür.


def feature_columns(df, feature_set):
    """İstenen feature ailesine göre kolon seçer."""
    prefix_map = {
        "morgan": ["Morgan_"],
        "maccs": ["MACCS_"],
        "rdkit": ["RDKit_"],
        "avalon": ["Avalon_"],
        "maccs_morgan": ["MACCS_", "Morgan_"],
        "maccs_rdkit": ["MACCS_", "RDKit_"],
        "morgan_rdkit": ["Morgan_", "RDKit_"],
        "all_available": ["Morgan_", "MACCS_", "RDKit_", "Avalon_"],
    }
    # Feature set adları ile kolon prefixleri eşleştirilir.
    prefixes = prefix_map[feature_set]
    # İstenen feature set için prefix listesi alınır.
    cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes)]
    # Seçilen prefixlerden biriyle başlayan kolonlar feature kolonu olarak alınır.
    if not cols:
        # Hiç feature kolonu bulunamazsa açık hata verilir.
        raise ValueError(f"{feature_set} için feature kolonu bulunamadı.")
    return cols
    # Seçilen feature kolon isimleri döndürülür.


def make_xy(df, cols):
    """Feature kolonlarından X matrisi ve Target kolonundan y vektörü üretir."""
    X = (
        df[cols]
        # Sadece seçilen feature kolonları alınır.
        .apply(pd.to_numeric, errors="coerce")
        # Sayısal olmayan değerler güvenli şekilde NaN yapılır.
        .replace([np.inf, -np.inf], np.nan)
        # Sonsuz değerler NaN'a çevrilir.
        .to_numpy(dtype=np.float32)
        # sklearn modelleri için numpy matrise çevrilir.
    )
    y = df["Target"].astype(int).to_numpy()
    # Binary target kolonu integer numpy vektörüne çevrilir.
    return X, y
    # Modelleme için X ve y döndürülür.


def split_data(df, cols):
    """Sınıf oranını koruyarak train/test ayrımı yapar."""
    X, y = make_xy(df, cols)
    # Seçilen featurelardan X ve y hazırlanır.
    return train_test_split(X, y, df, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    # Stratify ile sınıf oranı train/test içinde korunur.


def impute_train_test(X_train, X_test):
    """Eksik değerleri yalnızca train setten öğrenilen median değerlerle doldurur."""
    imputer = SimpleImputer(strategy="median")
    # SimpleImputer her feature için median değeri train setten öğrenir.
    X_train_imputed = imputer.fit_transform(X_train)
    # Imputer sadece train set üzerinde fit edilir; test bilgisi train'e sızmaz.
    X_test_imputed = imputer.transform(X_test)
    # Test set aynı imputer ile dönüştürülür.
    return X_train_imputed, X_test_imputed
    # Model eğitiminde kullanılacak temiz train/test matrisleri döndürülür.


def class1_score(model, X):
    """Modelden class 1 için skor veya olasılık üretir."""
    if hasattr(model, "predict_proba"):
        # Model olasılık üretebiliyorsa class 1 olasılığı alınır.
        score = model.predict_proba(X)[:, 1]
        # Class 1 olasılıkları alınır.
    elif hasattr(model, "decision_function"):
        # Olasılık yoksa karar fonksiyonu skoru kullanılır.
        score = model.decision_function(X)
        # Karar fonksiyonu skoru alınır.
    else:
        score = model.predict(X).astype(float)
        # Son seçenek olarak sınıf tahmini skor gibi kullanılır.

    score = np.asarray(score, dtype=float)
    # Skorlar numpy array formatına çevrilir.
    score = np.nan_to_num(score, nan=0.5, posinf=1.0, neginf=0.0)
    # ROC hesabı bozulmasın diye NaN/inf skorlar güvenli değerlere çekilir.
    return score
    # Temizlenmiş class 1 skoru döndürülür.


def metric_dict(y_true, y_pred, y_score):
    """Test tahminlerinden temel sınıflandırma metriklerini hesaplar."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    # Confusion matrix değerleri ayrı ayrı alınır.
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    # Specificity = TN / (TN + FP) olarak hesaplanır.
    return {
        "ROC": roc_auc_score(y_true, y_score),
        # ROC-AUC, pozitif ve negatif sınıf ayrım gücünü ölçer.
        "AP": average_precision_score(y_true, y_score),
        # AP, precision-recall eğrisi altındaki ortalama performanstır.
        "F1": f1_score(y_true, y_pred, zero_division=0),
        # F1, precision ve recall dengesini özetler.
        "Accuracy": accuracy_score(y_true, y_pred),
        # Accuracy, toplam doğru sınıflandırma oranıdır.
        "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
        # Balanced accuracy, sınıf dengesizliğinde daha adil bir doğruluk ölçüsüdür.
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        # Recall = TP / (TP + FN) olarak hesaplanır.
        "Specificity": specificity,
        # Specificity = TN / (TN + FP) olarak hesaplanır.
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        # Precision = TP / (TP + FP) olarak hesaplanır.
        "TN": int(tn),
        # True Negative sayısı.
        "FP": int(fp),
        # False Positive sayısı.
        "FN": int(fn),
        # False Negative sayısı.
        "TP": int(tp),
        # True Positive sayısı.
    }


def make_random_forest(**params):
    """RandomForest modeli kurar; parametre verilirse onları kullanır."""
    default_params = {
        "n_estimators": 300,
        # 300 ağaç, eğitim süresi ve stabilite arasında dengeli bir seçimdir.
        "max_features": "sqrt",
        # Her bölünmede featureların karekökü kadar aday denenir.
        "class_weight": "balanced_subsample",
        # Sınıf dengesizliğini her bootstrap örneğinde dengelemeye çalışır.
        "random_state": RANDOM_STATE,
        # Tekrarlanabilirlik sağlar.
        "n_jobs": -1,
        # Mevcut tüm işlemciler kullanılır.
    }
    # Varsayılan RF parametreleri tanımlanır.
    default_params.update(params)
    # Random search sonucundan gelen parametreler varsa varsayılanların üzerine yazılır.
    return RandomForestClassifier(**default_params)
    # RandomForest modeli döndürülür.


def plot_metric_bar(df, label_col, metric, title, out_file=None, top_n=None):
    """Performans sonuçlarını yatay bar chart olarak çizer."""
    plot_df = df.sort_values(metric, ascending=False).copy()
    # En iyi sonuçlar üstte olacak şekilde metrik değerine göre sıralanır.
    if top_n is not None:
        # Çok uzun tablolar için ilk n sonuç çizilebilir.
        plot_df = plot_df.head(top_n)
    plot_df = plot_df.sort_values(metric, ascending=True)
    # Yatay bar chartta en iyi değer üstte görünsün diye çizim öncesi ters sıralanır.

    plt.figure(figsize=(9, max(4, 0.35 * len(plot_df))))
    # Grafik boyutu satır sayısına göre ayarlanır.
    plt.barh(plot_df[label_col].astype(str), plot_df[metric])
    # Her senaryo için metrik değeri yatay bar olarak çizilir.
    plt.xlabel(metric)
    # X eksenine hangi metrik çizildiği yazılır.
    if metric == "ROC":
        # ROC-AUC farkları daha net görünsün diye ROC grafikleri 0.60'tan başlatılır.
        plt.xlim(left=0.60)
    plt.title(title)
    # Grafiğe açıklayıcı başlık eklenir.
    plt.tight_layout()
    # Etiketlerin kesilmemesi için yerleşim düzenlenir.
    if out_file:
        # Dosya yolu verilmişse grafik kaydedilir.
        Path(out_file).parent.mkdir(parents=True, exist_ok=True)
        # Kayıt klasörü yoksa oluşturulur.
        plt.savefig(out_file, dpi=300, bbox_inches="tight")
        # Grafik yüksek çözünürlüklü PNG olarak kaydedilir.
    plt.show()
    # Grafik ekranda gösterilir.


def add_gain(current_df, previous_df, current_name, previous_name):
    """Bir önceki adıma göre ROC/AP/F1 farkını ve yüzde değişimi hesaplar."""
    rows = []
    # Hesaplanan karşılaştırma satırları burada biriktirilir.
    for _, current in current_df.iterrows():
        # Mevcut adımın her veri seti sonucu tek tek dolaşılır.
        dataset = current["Dataset"]
        # Karşılaştırılacak veri seti adı alınır.
        prev = previous_df[previous_df["Dataset"] == dataset].sort_values("ROC", ascending=False).iloc[0]
        # Aynı veri setinin önceki adımdaki en iyi sonucu alınır.
        row = current.to_dict()
        # Mevcut sonuç satırı sözlüğe çevrilir.
        row["CurrentStep"] = current_name
        # Mevcut adım adı eklenir.
        row["ComparedWith"] = previous_name
        # Karşılaştırılan önceki adım adı eklenir.
        row["Previous_ROC"] = prev["ROC"]
        # Önceki ROC değeri tabloya eklenir.
        row["ROC_Delta"] = current["ROC"] - prev["ROC"]
        # Mutlak ROC farkı hesaplanır.
        row["ROC_Gain_%"] = 100 * (current["ROC"] - prev["ROC"]) / abs(prev["ROC"]) if abs(prev["ROC"]) > 1e-12 else np.nan
        # ROC yüzde değişimi hesaplanır.
        row["Previous_AP"] = prev["AP"]
        # Önceki AP değeri tabloya eklenir.
        row["AP_Delta"] = current["AP"] - prev["AP"]
        # AP farkı hesaplanır.
        row["AP_Gain_%"] = 100 * (current["AP"] - prev["AP"]) / abs(prev["AP"]) if abs(prev["AP"]) > 1e-12 else np.nan
        # AP yüzde değişimi hesaplanır.
        row["Previous_F1"] = prev["F1"]
        # Önceki F1 değeri tabloya eklenir.
        row["F1_Delta"] = current["F1"] - prev["F1"]
        # F1 farkı hesaplanır.
        row["F1_Gain_%"] = 100 * (current["F1"] - prev["F1"]) / abs(prev["F1"]) if abs(prev["F1"]) > 1e-12 else np.nan
        # F1 yüzde değişimi hesaplanır.
        rows.append(row)
        # Karşılaştırma satırı listeye eklenir.
    return pd.DataFrame(rows)
    # Tüm karşılaştırmalar tablo olarak döndürülür.


def resample_indices(y_train, ratio, method):
    """Train set üzerinde over/under sampling için seçilecek indeksleri üretir."""
    if method == "none":
        # Sampling yoksa train set olduğu gibi kullanılır.
        return np.arange(len(y_train))

    rng = np.random.RandomState(RANDOM_STATE)
    # Sampling işleminin tekrarlanabilir olması için sabit random generator oluşturulur.
    pos = np.where(y_train == 1)[0]
    # Pozitif sınıf indexleri alınır.
    neg = np.where(y_train == 0)[0]
    # Negatif sınıf indexleri alınır.
    current_ratio = len(pos) / len(neg)
    # Mevcut pozitif/negatif oranı hesaplanır.

    if method == "oversampling":
        # Oversampling az olan sınıfı çoğaltır.
        if current_ratio < ratio:
            n_pos, n_neg = int(np.ceil(ratio * len(neg))), len(neg)
            # Hedef oran için pozitif sınıf çoğaltılır.
        else:
            n_pos, n_neg = len(pos), int(np.ceil(len(pos) / ratio))
            # Hedef oran için negatif sınıf çoğaltılır.
        selected_pos = rng.choice(pos, n_pos, replace=n_pos > len(pos))
        # Pozitif örnekler gerekirse tekrarlı seçilir.
        selected_neg = rng.choice(neg, n_neg, replace=n_neg > len(neg))
        # Negatif örnekler gerekirse tekrarlı seçilir.

    elif method == "undersampling":
        # Undersampling fazla olan sınıfı azaltır.
        if current_ratio < ratio:
            n_pos = len(pos)
            # Pozitif sınıf korunur.
            n_neg = min(len(neg), max(5, int(np.floor(len(pos) / ratio))))
            # Negatif sınıf hedef orana göre azaltılır.
        else:
            n_pos = min(len(pos), max(5, int(np.floor(ratio * len(neg)))))
            # Pozitif sınıf hedef orana göre azaltılır.
            n_neg = len(neg)
            # Negatif sınıf korunur.
        selected_pos = rng.choice(pos, n_pos, replace=False)
        # Pozitif sınıf tekrarsız örneklenir.
        selected_neg = rng.choice(neg, n_neg, replace=False)
        # Negatif sınıf tekrarsız örneklenir.

    else:
        raise ValueError("method: none, oversampling veya undersampling olmalı.")
        # Bilinmeyen sampling yöntemi için açık hata verilir.

    idx = np.concatenate([selected_pos, selected_neg])
    # Pozitif ve negatif indexler birleştirilir.
    rng.shuffle(idx)
    # Train sırası karıştırılır.
    return idx
    # Sampling uygulanmış train indexleri döndürülür.

## 4. Sampling gatekeeper tanımı

Step 6 sonucunda küçük ROC-AUC artışı veren sampling senaryoları yeni gatekeeper olarak alınır:

- `ERa_BLA_assay`: `balanced_oversampling`
- `ERa_LUC_VM7_assay`: `negative_5x_oversampling`

Bu hücrede bu gatekeeper senaryoları yeniden eğitilir. Aynı split kullanılır; sampling sadece train set üzerinde yapılır.

In [ ]:
forced_feature_sets = {
    "ERa_BLA_assay": "rdkit",
    "ERa_LUC_VM7_assay": "all_available",
}
# Step 3 sonucuna göre devam edilecek feature setler sabitlenir.

forced_sampling_scenarios = {
    "ERa_BLA_assay": ("balanced_oversampling", 1.0, "oversampling"),
    "ERa_LUC_VM7_assay": ("negative_5x_oversampling", 0.2, "oversampling"),
}
# Step 6 sonucuna göre devam edilecek sampling senaryoları sabitlenir.

gatekeeper_rows = []
# Sampling gatekeeper sonuçları burada tutulur.

prepared_data = {}
# Her veri setinin split, imputed matrix ve sampling bilgileri tuning hücresinde tekrar kullanılmak üzere saklanır.

for dataset_key in DATASETS:
    # Her veri seti sırayla işlenir.
    df = load_feature_table(dataset_key)
    # Hazır feature CSV dosyası okunur.

    selected_feature_set = forced_feature_sets[dataset_key]
    # Bu veri seti için kullanılacak feature set alınır.
    selected_features = feature_columns(df, selected_feature_set)
    # Seçilen feature setin kolonları alınır.

    scenario_name, ratio, sampling_method = forced_sampling_scenarios[dataset_key]
    # Step 6'dan gelen en iyi sampling senaryosu alınır.

    X_train, X_test, y_train, y_test, df_train, df_test = split_data(df, selected_features)
    # Aynı random_state ile train/test ayrımı yapılır.

    X_train, X_test = impute_train_test(X_train, X_test)
    # Eksik değerler SimpleImputer ile doldurulur.
    # Imputer median değerleri sadece train setten öğrenir ve sonra test sete uygular.

    sampled_idx = resample_indices(y_train, ratio=ratio, method=sampling_method)
    # Sampling indexleri sadece train set üzerinden üretilir.

    X_train_sampled = X_train[sampled_idx]
    # Sampling uygulanmış train feature matrisi hazırlanır.
    y_train_sampled = y_train[sampled_idx]
    # Sampling uygulanmış train target vektörü hazırlanır.

    sampled_counts = dict(pd.Series(y_train_sampled).value_counts().sort_index())
    # Sampling sonrası train sınıf dağılımı hesaplanır.
    print(f"{dataset_key} | gatekeeper sampling: {scenario_name} | sampled train distribution: {sampled_counts}")
    # Sampling sonrası train dağılımı ekrana yazdırılır.

    model = make_random_forest()
    # Sampling gatekeeper için RandomForest modeli kurulur.
    model.fit(X_train_sampled, y_train_sampled)
    # Model sampling uygulanmış train set üzerinde eğitilir.

    y_pred = model.predict(X_test)
    # Doğal test seti için sınıf tahmini alınır.
    y_score = class1_score(model, X_test)
    # ROC/AP için class 1 skoru alınır.

    metrics = metric_dict(y_test, y_pred, y_score)
    # Test performans metrikleri hesaplanır.
    metrics.update({
        "Dataset": dataset_key,
        "Comparison": "RF sampling gatekeeper",
        "Step": "06_sampling_gatekeeper",
        "ModelType": "RandomForest",
        "FeatureSet": selected_feature_set,
        "SamplingScenario": scenario_name,
        "SamplingMethod": sampling_method,
        "SamplingRatio": ratio,
        "TrainClass0_after_sampling": int(sampled_counts.get(0, 0)),
        "TrainClass1_after_sampling": int(sampled_counts.get(1, 0)),
        "FeatureCount": len(selected_features),
        "SelectedFeatures": selected_features,
    })
    # Sonuç satırına gatekeeper feature ve sampling bilgileri eklenir.

    gatekeeper_rows.append(metrics)
    # Gatekeeper sonucu listeye eklenir.

    prepared_data[dataset_key] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "X_train_sampled": X_train_sampled,
        "y_train_sampled": y_train_sampled,
        "selected_features": selected_features,
        "feature_set": selected_feature_set,
        "scenario_name": scenario_name,
        "sampling_method": sampling_method,
        "sampling_ratio": ratio,
        "sampled_counts": sampled_counts,
    }
    # Tuning hücresinde aynı split ve aynı sampling kullanılabilsin diye veri saklanır.

gatekeeper_df = pd.DataFrame(gatekeeper_rows)
# Sampling gatekeeper sonuçları tabloya çevrilir.

show_table(
    gatekeeper_df[["Dataset", "ModelType", "FeatureSet", "SamplingScenario", "FeatureCount", "ROC", "AP", "F1"]],
    title="RandomForest sampling gatekeeper sonuçları"
)
# Tuning öncesi gatekeeper performansı ekranda gösterilir.

## 5. RandomForest random search tuning

Bu adımda yalnızca RandomForest için hiperparametre araması yapılır.

`RandomizedSearchCV` arama uzayından 10 aday kombinasyon seçer. Bu adaylar önce train set içindeki 3-fold CV ile sıralanır; ardından her aday model sampling uygulanmış train set üzerinde yeniden eğitilir ve doğal test set üzerinde ayrı ayrı değerlendirilir.

Bu notebookta ana aday tablolarında iç CV skoru gösterilmez. Asıl gösterilen değerler test set performansıdır:

- `Test_ROC`
- `Test_AP`
- `Test_F1`
- `Test_BalancedAccuracy`

Her veri seti/pipeline ayrı blokta gösterilir. Bloklar arasında `##########` ayırıcı bulunur.


In [ ]:
lesson_out = OUTPUT_ROOT / "07_random_search_tuning_fast"
# Hızlı tuning sonuçları için çıktı klasörü belirlenir.
lesson_out.mkdir(parents=True, exist_ok=True)
# Çıktı klasörü yoksa oluşturulur.

param_distributions = {
    "n_estimators": [100, 150, 200, 250, 300],
    # Ağaç sayısı için 5 makul seçenek kullanılır.
    # RandomizedSearchCV bunların tamamını sistematik taramaz; N_RANDOM_SEARCH_ITER kadar rastgele kombinasyon seçer.
    "max_depth": [None, 8, 15, 25],
    # Ağaç derinliği için kısa ama anlamlı bir liste kullanılır.
    "min_samples_split": [2, 5, 10],
    # Bir node'u bölmek için gereken minimum örnek sayısı için 3 seçenek kullanılır.
    "min_samples_leaf": [1, 2, 4, 6, 8],
    # Yaprak node için minimum örnek sayısı 5 seçenekle sınırlandırılır.
    "max_features": ["sqrt", "log2", 0.50],
    # Her splitte bakılacak feature sayısı/oranı için 3 seçenek kullanılır.
    "bootstrap": [True],
    # Bootstrap açık bırakılır; arama süresini ve varyasyonu kontrol altında tutar.
    "class_weight": ["balanced", "balanced_subsample"],
    # Sınıf dengesizliği olduğu için iki dengeli ağırlık stratejisi denenir.
    "criterion": ["gini", "entropy"],
    # Split kalitesi için iki temel ölçüt denenir.
}
# RandomForest için kısa ders/Colab kullanımına uygun random search alanı tanımlanır.


def print_pipeline_separator(dataset_key):
    """İki classification pipeline sonucunu ekranda net şekilde ayırır."""
    print("\n" + "#" * 100)
    # Birinci ve ikinci classification çıktıları karışmasın diye ayraç basılır.
    print(f"PIPELINE: {dataset_key}")
    # Hangi veri setinin raporlandığı açıkça yazılır.
    print("#" * 100)
    # Ayraç tamamlanır.


def short_param_label(params, rank):
    """RF parametrelerini bar chart etiketi için kısa hale getirir."""
    return (
        f"rank {rank} | "
        f"n={params.get('n_estimators')} | "
        f"depth={params.get('max_depth')} | "
        f"leaf={params.get('min_samples_leaf')} | "
        f"feat={params.get('max_features')}"
    )
    # Çok uzun parametre sözlüğü yerine okunabilir kısa etiket döndürülür.


def make_rf_from_params(params):
    """Random search aday parametrelerinden RandomForest modeli oluşturur."""
    clean_params = dict(params)
    # Parametre sözlüğü kopyalanır.
    clean_params["random_state"] = RANDOM_STATE
    # Tekrarlanabilirlik için random_state sabitlenir.
    clean_params["n_jobs"] = -1
    # RandomForest paralel çalıştırılır.
    return RandomForestClassifier(**clean_params)
    # Aday RandomForest modeli döndürülür.


def plot_candidate_test_bar(candidate_table, dataset_key, out_file):
    """Random search RF adaylarının test ROC-AUC değerlerini bar chart olarak çizer."""
    plot_df = candidate_table.sort_values("Test_ROC", ascending=True).copy()
    # Yatay bar chart için adaylar test ROC-AUC değerine göre sıralanır.

    plt.figure(figsize=(10, max(4, 0.45 * len(plot_df))))
    # Grafik boyutu aday sayısına göre ayarlanır.
    plt.barh(plot_df["CandidateLabel"], plot_df["Test_ROC"])
    # Her RF adayının test ROC-AUC değeri yatay bar olarak çizilir.
    plt.xlabel("Test ROC-AUC")
    # X ekseninde doğal test set üzerindeki ROC-AUC gösterilir.
    plt.xlim(left=0.60)
    # ROC farkları daha net görünsün diye grafik 0.60'tan başlatılır.
    plt.title(f"{dataset_key} — Random search RF adayları: test ROC-AUC")
    # Grafik başlığı yazılır.
    plt.tight_layout()
    # Etiketlerin kesilmemesi için yerleşim düzenlenir.
    Path(out_file).parent.mkdir(parents=True, exist_ok=True)
    # Grafik klasörü yoksa oluşturulur.
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    # Grafik yüksek çözünürlüklü PNG olarak kaydedilir.
    plt.show()
    # Grafik ekranda gösterilir.


tuning_rows = []
# En iyi test performansı veren random search adayı burada tutulur.
all_candidate_rows = []
# Random search içinde denenen RF adaylarının test sonuçları burada tutulur.
comparison_rows = []
# Gatekeeper vs en iyi random search adayı karşılaştırma satırları burada tutulur.

for dataset_key in DATASETS:
    # Her veri seti sırayla işlenir.
    print_pipeline_separator(dataset_key)
    # Birinci ve ikinci classification çıktıları ayrı bloklar halinde gösterilir.

    data = prepared_data[dataset_key]
    # Gatekeeper hücresinde hazırlanmış split ve sampling bilgileri alınır.

    X_train_sampled = data["X_train_sampled"]
    # Sampling uygulanmış train feature matrisi alınır.
    y_train_sampled = data["y_train_sampled"]
    # Sampling uygulanmış train target vektörü alınır.
    X_test = data["X_test"]
    # Doğal test feature matrisi alınır.
    y_test = data["y_test"]
    # Doğal test target vektörü alınır.

    base_rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
    # Random search için temel RandomForest modeli kurulur.

    search = RandomizedSearchCV(
        estimator=base_rf,
        # Tune edilecek temel model RandomForest'tır.
        param_distributions=param_distributions,
        # Daraltılmış arama alanı verilir.
        n_iter=N_RANDOM_SEARCH_ITER,
        # Kaç farklı parametre kombinasyonu deneneceği belirlenir.
        # Bu sürümde 10 kombinasyon denenir.
        scoring="roc_auc",
        # Random search iç seçimi ROC-AUC ile yapılır.
        cv=CV_FOLDS,
        # Cross-validation fold sayısı belirlenir.
        random_state=RANDOM_STATE,
        # Random search tekrarlanabilir hale getirilir.
        n_jobs=-1,
        # Mevcut işlemciler kullanılır.
        verbose=1,
        # Arama ilerleyişi ekrana yazdırılır.
        error_score=np.nan,
        # Hata veren kombinasyon olursa süreç tamamen durmaz.
    )
    # RandomizedSearchCV nesnesi oluşturulur.

    search.fit(X_train_sampled, y_train_sampled)
    # Random search sampling uygulanmış train set üzerinde çalıştırılır.
    # İç CV skoru model seçimi için kullanılır; ana sonuç tablosunda gösterilmez.

    candidate_df = pd.DataFrame(search.cv_results_)
    # Random search sırasında denenen bütün parametre kombinasyonları tabloya çevrilir.
    candidate_df = candidate_df.sort_values("rank_test_score").reset_index(drop=True)
    # Adaylar iç CV sıralamasına göre dizilir; sonra her aday test set üzerinde ayrıca değerlendirilecektir.

    dataset_candidate_rows = []
    # Bu veri setinde denenen RF adayları ayrı tabloda gösterilmek üzere tutulur.

    for candidate_id, candidate in enumerate(candidate_df.itertuples(index=False), start=1):
        # Random search adayları tek tek dolaşılır.
        candidate_params = dict(getattr(candidate, "params"))
        # Aday parametreleri sözlük olarak alınır.

        candidate_model = make_rf_from_params(candidate_params)
        # Aday parametrelerle yeni RandomForest modeli oluşturulur.
        candidate_model.fit(X_train_sampled, y_train_sampled)
        # Aday model sampling uygulanmış train set üzerinde yeniden eğitilir.

        y_pred = candidate_model.predict(X_test)
        # Doğal test seti için sınıf tahmini alınır.
        y_score = class1_score(candidate_model, X_test)
        # Doğal test seti için class 1 skoru alınır.

        test_metrics = metric_dict(y_test, y_pred, y_score)
        # Aday modelin doğal test set metrikleri hesaplanır.

        candidate_row = {
            "Dataset": dataset_key,
            "ModelType": "RandomForest",
            "FeatureSet": data["feature_set"],
            "SamplingScenario": data["scenario_name"],
            "SamplingMethod": data["sampling_method"],
            "SamplingRatio": data["sampling_ratio"],
            "RandomSearchCandidateID": candidate_id,
            "Test_ROC": test_metrics["ROC"],
            "Test_AP": test_metrics["AP"],
            "Test_F1": test_metrics["F1"],
            "Test_Accuracy": test_metrics["Accuracy"],
            "Test_BalancedAccuracy": test_metrics["BalancedAccuracy"],
            "Test_Recall": test_metrics["Recall"],
            "Test_Specificity": test_metrics["Specificity"],
            "Test_Precision": test_metrics["Precision"],
            "TN": test_metrics["TN"],
            "FP": test_metrics["FP"],
            "FN": test_metrics["FN"],
            "TP": test_metrics["TP"],
            "n_estimators": candidate_params.get("n_estimators"),
            "max_depth": candidate_params.get("max_depth"),
            "min_samples_split": candidate_params.get("min_samples_split"),
            "min_samples_leaf": candidate_params.get("min_samples_leaf"),
            "max_features": candidate_params.get("max_features"),
            "bootstrap": candidate_params.get("bootstrap"),
            "class_weight": candidate_params.get("class_weight"),
            "criterion": candidate_params.get("criterion"),
            "Params": str(candidate_params),
        }
        # Aday modelin test performansı ve hiperparametreleri tek satırda saklanır.

        dataset_candidate_rows.append(candidate_row)
        # Bu veri setinin aday listesine eklenir.

    dataset_candidate_table = pd.DataFrame(dataset_candidate_rows)
    # Bu veri setinin RF adayları tabloya çevrilir.
    dataset_candidate_table = dataset_candidate_table.sort_values("Test_ROC", ascending=False).reset_index(drop=True)
    # Adaylar test ROC-AUC değerine göre sıralanır.
    dataset_candidate_table["CandidateRank"] = np.arange(1, len(dataset_candidate_table) + 1)
    # Test ROC-AUC sırasına göre aday rank değeri atanır.
    dataset_candidate_table["CandidateLabel"] = [
        short_param_label(params=eval(p), rank=r)
        for p, r in zip(dataset_candidate_table["Params"], dataset_candidate_table["CandidateRank"])
    ]
    # Bar chart için okunabilir aday etiketi oluşturulur.

    dataset_candidate_table.to_csv(lesson_out / f"{dataset_key}_random_search_candidate_test_performance.csv", index=False)
    # Veri setine özel RF aday test performansı CSV olarak kaydedilir.
    dataset_candidate_table.to_csv(lesson_out / f"{dataset_key}_all_random_search_rf_candidates.csv", index=False)
    # Önceki dosya adıyla uyumluluk için aynı tablo ayrıca kaydedilir.

    all_candidate_rows.extend(dataset_candidate_table.to_dict("records"))
    # Genel candidate listesine bu veri setinin bütün adayları eklenir.

    print_pipeline_separator(dataset_key)
    # Aday performans tablosu öncesinde veri seti ayıracı tekrar basılır.

    show_table(
        dataset_candidate_table[
            [
                "Dataset",
                "ModelType",
                "FeatureSet",
                "SamplingScenario",
                "CandidateRank",
                "Test_ROC",
                "Test_AP",
                "Test_F1",
                "Test_BalancedAccuracy",
                "Test_Recall",
                "Test_Specificity",
                "Test_Precision",
                "TN",
                "FP",
                "FN",
                "TP",
                "n_estimators",
                "max_depth",
                "min_samples_split",
                "min_samples_leaf",
                "max_features",
                "class_weight",
                "criterion",
                "Params",
            ]
        ],
        n=20,
        title=f"{dataset_key} — random search adaylarının doğal test set performansı"
    )
    # Bu veri setinde denenen RF adaylarının test performansı ayrı tablo olarak gösterilir.

    plot_candidate_test_bar(
        dataset_candidate_table,
        dataset_key,
        lesson_out / f"{dataset_key}_random_search_candidate_test_roc.png",
    )
    # Bu veri setinde denenen RF adaylarının test ROC-AUC değerleri bar chart olarak çizilir.

    best_candidate = dataset_candidate_table.iloc[0].to_dict()
    # Doğal test ROC-AUC'ye göre en iyi RF adayı seçilir.

    metrics = {
        "ROC": best_candidate["Test_ROC"],
        "AP": best_candidate["Test_AP"],
        "F1": best_candidate["Test_F1"],
        "Accuracy": best_candidate["Test_Accuracy"],
        "BalancedAccuracy": best_candidate["Test_BalancedAccuracy"],
        "Recall": best_candidate["Test_Recall"],
        "Specificity": best_candidate["Test_Specificity"],
        "Precision": best_candidate["Test_Precision"],
        "TN": best_candidate["TN"],
        "FP": best_candidate["FP"],
        "FN": best_candidate["FN"],
        "TP": best_candidate["TP"],
    }
    # En iyi adayın test metrikleri standart formatta alınır.

    metrics.update({
        "Dataset": dataset_key,
        "Comparison": "Best random-search RF candidate",
        "Step": "07_random_search_tuning",
        "ModelType": "RandomForest",
        "FeatureSet": data["feature_set"],
        "SamplingScenario": data["scenario_name"],
        "SamplingMethod": data["sampling_method"],
        "SamplingRatio": data["sampling_ratio"],
        "FeatureCount": len(data["selected_features"]),
        "SelectedCandidateRank": int(best_candidate["CandidateRank"]),
        "BestParams": best_candidate["Params"],
        "RandomSearchIterations": N_RANDOM_SEARCH_ITER,
        "CVFoldsUsedForSearch": CV_FOLDS,
    })
    # Tuning sonucuna model, sampling ve parametre bilgileri eklenir.
    # İç CV skoru burada raporlanmaz; final karşılaştırma doğal test set performansıdır.

    tuning_rows.append(metrics)
    # Tuning sonucu listeye eklenir.

    gatekeeper_row = gatekeeper_df[gatekeeper_df["Dataset"] == dataset_key].iloc[0]
    # Aynı veri setinin sampling gatekeeper sonucu alınır.

    reference_row = {
        "Dataset": dataset_key,
        "Comparison": "RF sampling gatekeeper",
        "ModelType": "RandomForest",
        "FeatureSet": gatekeeper_row["FeatureSet"],
        "SamplingScenario": gatekeeper_row["SamplingScenario"],
        "SamplingMethod": gatekeeper_row["SamplingMethod"],
        "SamplingRatio": gatekeeper_row["SamplingRatio"],
        "FeatureCount": int(gatekeeper_row["FeatureCount"]),
        "ROC": gatekeeper_row["ROC"],
        "AP": gatekeeper_row["AP"],
        "F1": gatekeeper_row["F1"],
        "ROC_Delta_vs_Gatekeeper": 0.0,
        "ROC_Gain_%": 0.0,
        "AP_Delta_vs_Gatekeeper": 0.0,
        "AP_Gain_%": 0.0,
        "F1_Delta_vs_Gatekeeper": 0.0,
        "F1_Gain_%": 0.0,
    }
    # Karşılaştırma tablosunun ilk satırı sampling gatekeeper sonucudur.

    tuned_row = {
        "Dataset": dataset_key,
        "Comparison": "Best random-search RF candidate",
        "ModelType": "RandomForest",
        "FeatureSet": metrics["FeatureSet"],
        "SamplingScenario": metrics["SamplingScenario"],
        "SamplingMethod": metrics["SamplingMethod"],
        "SamplingRatio": metrics["SamplingRatio"],
        "FeatureCount": int(metrics["FeatureCount"]),
        "ROC": metrics["ROC"],
        "AP": metrics["AP"],
        "F1": metrics["F1"],
        "ROC_Delta_vs_Gatekeeper": metrics["ROC"] - gatekeeper_row["ROC"],
        "ROC_Gain_%": 100 * (metrics["ROC"] - gatekeeper_row["ROC"]) / abs(gatekeeper_row["ROC"]) if abs(gatekeeper_row["ROC"]) > 1e-12 else np.nan,
        "AP_Delta_vs_Gatekeeper": metrics["AP"] - gatekeeper_row["AP"],
        "AP_Gain_%": 100 * (metrics["AP"] - gatekeeper_row["AP"]) / abs(gatekeeper_row["AP"]) if abs(gatekeeper_row["AP"]) > 1e-12 else np.nan,
        "F1_Delta_vs_Gatekeeper": metrics["F1"] - gatekeeper_row["F1"],
        "F1_Gain_%": 100 * (metrics["F1"] - gatekeeper_row["F1"]) / abs(gatekeeper_row["F1"]) if abs(gatekeeper_row["F1"]) > 1e-12 else np.nan,
        "SelectedCandidateRank": metrics["SelectedCandidateRank"],
        "BestParams": metrics["BestParams"],
    }
    # Karşılaştırma tablosunun ikinci satırı en iyi random-search RF adayıdır.

    comparison_rows.extend([reference_row, tuned_row])
    # İki satırlık karşılaştırma genel listeye eklenir.

tuning_df = pd.DataFrame(tuning_rows).sort_values("ROC", ascending=False)
# En iyi aday sonuçları tabloya çevrilir ve ROC değerine göre sıralanır.
all_candidate_df = pd.DataFrame(all_candidate_rows).sort_values(["Dataset", "CandidateRank"])
# Random search içinde denenen bütün RF adaylarının test sonuçları tabloya çevrilir.
comparison_df = pd.DataFrame(comparison_rows)
# Gatekeeper ve en iyi random-search RF adayı karşılaştırmaları tabloya çevrilir.

tuning_df.to_csv(lesson_out / "07_rf_tuning_metrics.csv", index=False)
# En iyi aday sonuçları CSV olarak kaydedilir.
all_candidate_df.to_csv(lesson_out / "07_all_random_search_rf_candidates.csv", index=False)
# Random search içinde denenen bütün RF adaylarının test sonuçları CSV olarak kaydedilir.
all_candidate_df.to_csv(lesson_out / "07_all_random_search_rf_candidate_test_performance.csv", index=False)
# Açık isimli aday test performansı CSV olarak ayrıca kaydedilir.
comparison_df.to_csv(lesson_out / "07_sampling_gatekeeper_vs_tuned_rf_all.csv", index=False)
# Gatekeeper ve en iyi random-search RF adayı karşılaştırmaları CSV olarak kaydedilir.

for dataset_key in DATASETS:
    # İki classification veri seti için ayrı gatekeeper vs random-search RF tablosu üretilir.
    print_pipeline_separator(dataset_key)
    # Final karşılaştırma tabloları da pipeline bazında ayrı gösterilir.

    dataset_comparison_table = comparison_df[comparison_df["Dataset"] == dataset_key].copy()
    # İlgili veri setinin iki satırlık karşılaştırması alınır.
    dataset_comparison_table.to_csv(lesson_out / f"{dataset_key}_sampling_gatekeeper_vs_random_search_rf.csv", index=False)
    # Veri setine özel karşılaştırma tablosu CSV olarak kaydedilir.

    plot_metric_bar(
        dataset_comparison_table,
        label_col="Comparison",
        metric="ROC",
        title=f"{dataset_key} — RF sampling gatekeeper vs random-search RF",
        out_file=lesson_out / f"{dataset_key}_rf_sampling_gatekeeper_vs_random_search_rf_roc.png",
    )
    # Gatekeeper ve random-search RF ROC değerleri bar chart olarak çizilir.

    show_table(
        dataset_comparison_table[
            [
                "Dataset",
                "Comparison",
                "ModelType",
                "FeatureSet",
                "SamplingScenario",
                "SamplingMethod",
                "SamplingRatio",
                "FeatureCount",
                "ROC",
                "AP",
                "F1",
                "ROC_Delta_vs_Gatekeeper",
                "ROC_Gain_%",
                "AP_Delta_vs_Gatekeeper",
                "AP_Gain_%",
                "F1_Delta_vs_Gatekeeper",
                "F1_Gain_%",
            ]
        ],
        title=f"{dataset_key} — RF sampling gatekeeper vs random-search RF"
    )
    # Her veri seti için gatekeeper ve random-search RF karşılaştırması ekranda gösterilir.

gain_df = add_gain(tuning_df, gatekeeper_df, "07_RF_Random_Search_Test_Selected", "06_RF_Sampling_Gatekeeper")
# En iyi random-search RF adayı sampling gatekeeper sonucu ile karşılaştırılır.
gain_df.to_csv(lesson_out / "07_gain_vs_sampling_gatekeeper.csv", index=False)
# Kazanç tablosu CSV olarak kaydedilir.

print("\n" + "#" * 100)
# Özet tabloların başladığını ayırmak için başlık çizgisi basılır.
print("ÖZET — RANDOM SEARCH TUNING")
# Final özet başlığı yazdırılır.
print("#" * 100)
# Başlık çizgisi tamamlanır.

show_table(
    gain_df[["Dataset", "ModelType", "SamplingScenario", "ROC", "Previous_ROC", "ROC_Delta", "ROC_Gain_%", "AP", "F1", "RandomSearchIterations", "SelectedCandidateRank"]],
    title="07 — RF sampling gatekeeper sonucuna göre random-search kazancı"
)
# Random-search adımının doğal test set performans kazancı ekranda gösterilir.

show_table(
    tuning_df[["Dataset", "ModelType", "FeatureSet", "SamplingScenario", "SelectedCandidateRank", "ROC", "AP", "F1", "BalancedAccuracy", "BestParams"]],
    title="Random search tuning detayları — doğal test set üzerinde en iyi aday"
)
# En iyi RF adayının test performansı ve hiperparametreleri ekranda gösterilir.
